In [12]:
import forte2
import numpy as np
import matplotlib.pyplot as plt
#Using the original mutual correlation code. Using 'mod' versions would set the 1-rdm to zero for M2 analysis, 
# so the decomposition doesn't sum to the total energy
from mutual_correlation_energy import fragment_decomposition_energy_enumerated
from mutual_correlation_energy import onefrag_correlation_energy_enumerated, twofrag_correlation_energy_enumerated, threefrag_correlation_energy_enumerated, fourfrag_correlation_energy_enumerated

In [13]:
xyz = """
H 0.000 0.000 0.000
H 0.000 0.000 0.740
"""
system = forte2.System(xyz=xyz, basis_set="cc-pVTZ", auxiliary_basis_set="def2-universal-JKFIT")

rhf = forte2.RHF(charge=0)(system)
rhf.run()

ci=forte2.CI(forte2.State(system=system, multiplicity=1,ms=0.0),active_orbitals=[0,1,2,3,4,5,6,7])(rhf)
ci.run()

orbital_fragments = [[i] for i in ci.mo_space.active_indices]

result = fragment_decomposition_energy_enumerated(ci, orbital_fragments)

print("CI energy:           ", result["ci_energy"])
print("Decomposition energy:", result["decomposition_energy"])
print("Residual:            ", result["residual"])

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   H   0.00000000   0.00000000   0.00000000
   H   0.00000000   0.00000000   1.39839733
Parsed 2 atoms with basis set of 28 functions.
  Max eigenvalue: 4.217e+00
  Min eigenvalue: 1.591e-03
  Condition number: 2.650e+03
  Inverse condition number: 3.774e-04
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 28
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 1.591e-03
Number of electrons: 2
Number of alpha electrons: 1
Number of beta electrons: 1
Ms: 0
Total charge: 0
Number of basis functions: 28
Number of orthogonalized basis functions: 28
Number of auxiliary basis functions: 36
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 28
Number of auxiliary basis functions:

   2      -1.132975813680  -4.2647e-05   2.7065e-03   8.5290e-03    0.00000   S/E
   3      -1.132975939950  -1.2627e-07   2.5102e-04   3.8433e-04    0.00000   S/E
   4      -1.132975940364  -4.1471e-10   2.1413e-05   2.0606e-05    0.00000   S/E
   5      -1.132975940366  -1.3278e-12   1.2740e-06   1.0922e-06    0.00000   S/E
   6      -1.132975940366  -1.7764e-14   1.4310e-07   1.3731e-07    0.00000   S/E
RHF iterations converged

Final RHF Energy:      -1.132975940366
RHF time: 0.16 seconds
---------------------
Orbital Energies [Eh]
---------------------
Doubly Occupied:

0    (a) -0.594693    

Virtual:

1    (a) 0.167459     2    (a) 0.305425     3    (a) 0.644218     4    (a) 0.694039     5    (a) 0.694039     
6    (a) 1.098742     7    (a) 1.120359     8    (a) 1.120359     9    (a) 1.530730     10   (a) 2.338060     
11   (a) 2.412078     12   (a) 3.180484     13   (a) 3.180484     14   (a) 3.414147     15   (a) 3.414147     
16   (a) 3.677068     17   (a) 3.893525     18   (a

In [14]:
frag_orbs = [0,1,2,3,4,5,6,7]

ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=(),
)

ecore = ints.E #core energy
print(ecore)
e1=0
e2=0
e3=0
e4=0

for i in frag_orbs:
    e1 += onefrag_correlation_energy_enumerated(ci, [i])
    for j in frag_orbs:
        if j<i:
            e2 += twofrag_correlation_energy_enumerated(ci, [i], [j])
            for k in frag_orbs:
                if k<j:
                    e3 += threefrag_correlation_energy_enumerated(ci, [i], [j], [k])
                    for l in frag_orbs:
                        if l<k:
                            e4 += fourfrag_correlation_energy_enumerated(ci, [i], [j], [k], [l])

print(f"Total one-orbital energy term: {e1} EH")
print(f"Total two-orbital energy term: {e2} EH")
print(f"Total three-orbital energy term: {e3} EH")
print(f"Total four-orbital energy term: {e4} EH")
print(f"Total energy from Decomposition Analysis: {ecore+e1+e2+e3+e4}")
print(f"Total CI energy: {ci.E[0]}")

0.7151043390581081
Total one-orbital energy term: -1.8259030093703732 EH
Total two-orbital energy term: -0.033773624272326636 EH
Total three-orbital energy term: -0.011658072552282945 EH
Total four-orbital energy term: -8.435596010033028e-05 EH
Total energy from Decomposition Analysis: -1.156314723096975
Total CI energy: -1.1563147230969735


In [15]:
xyz = """
Be 0.000 0.000 0.000
"""
system = forte2.System(xyz=xyz, basis_set="cc-pVTZ", auxiliary_basis_set="def2-universal-JKFIT")

rhf = forte2.RHF(charge=0)(system)
rhf.run()

avas=forte2.AVAS(subspace=["Be(1s)","Be(2s)","Be(2p)"],
                 selection_method="separate",
                 num_active_docc=2,
                 num_active_uocc=3)(rhf)

ci=forte2.CI(forte2.State(system=system, multiplicity=1,ms=0.0))(avas)
ci.run()

orbital_fragments = [[i] for i in ci.mo_space.active_indices]

result = fragment_decomposition_energy_enumerated(ci, orbital_fragments, core_orbitals=ci.core_indices)

print("CI energy:           ", result["ci_energy"])
print("Decomposition energy:", result["decomposition_energy"])
print("Residual:            ", result["residual"])

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   BE   0.00000000   0.00000000   0.00000000
Parsed 1 atoms with basis set of 30 functions.
  Max eigenvalue: 2.362e+00
  Min eigenvalue: 3.547e-02
  Condition number: 6.659e+01
  Inverse condition number: 1.502e-02
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 30
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 3.547e-02
Number of electrons: 4
Number of alpha electrons: 2
Number of beta electrons: 2
Ms: 0
Total charge: 0
Number of basis functions: 30
Number of orthogonalized basis functions: 30
Number of auxiliary basis functions: 51
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.00 GB (doubled due to storing B_nPm)
Number of system basis functions: 30
Number of auxiliary basis functions: 51
Iter               Energy           ΔE 

In [16]:
frag_orbs = ci.active_indices

ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=ci.core_indices,
)

ecore = ints.E #core energy
print(ecore)
e1=0
e2=0
e3=0
e4=0

for i in frag_orbs:
    e1 += onefrag_correlation_energy_enumerated(ci, [i], core_orbitals=ci.core_indices)
    for j in frag_orbs:
        if j<i:
            e2 += twofrag_correlation_energy_enumerated(ci, [i], [j], core_orbitals=ci.core_indices)
            for k in frag_orbs:
                if k<j:
                    e3 += threefrag_correlation_energy_enumerated(ci, [i], [j], [k], core_orbitals=ci.core_indices)
                    for l in frag_orbs:
                        if l<k:
                            e4 += fourfrag_correlation_energy_enumerated(ci, [i], [j], [k], [l], core_orbitals=ci.core_indices)

print(f"Total one-orbital energy term: {e1} EH")
print(f"Total two-orbital energy term: {e2} EH")
print(f"Total three-orbital energy term: {e3} EH")
print(f"Total four-orbital energy term: {e4} EH")
print(f"Total energy from Decomposition Analysis: {ecore+e1+e2+e3+e4}")
print(f"Total CI energy: {ci.E[0]}")

0.0
Total one-orbital energy term: -16.4067080530835 EH
Total two-orbital energy term: 1.7936211860741995 EH
Total three-orbital energy term: -0.00017200417423967793 EH
Total four-orbital energy term: -4.120215416116181e-35 EH
Total energy from Decomposition Analysis: -14.613258871183538
Total CI energy: -14.613258871183538


In [23]:
xyz = """
N 0.000 0.000 0.000
N 0.000 0.000 1.0977
"""
system = forte2.System(xyz=xyz, basis_set="cc-pVTZ", auxiliary_basis_set="def2-universal-JKFIT")

rhf = forte2.RHF(charge=0)(system)
rhf.run()

avas=forte2.AVAS(subspace=["N(2s)","N(2p)"],
                 selection_method="separate",
                 num_active_docc=5,
                 num_active_uocc=3)(rhf)

ci=forte2.CI(forte2.State(system=system, multiplicity=1,ms=0.0))(avas)
ci.run()

orbital_fragments = [[i] for i in ci.mo_space.active_indices]

result = fragment_decomposition_energy_enumerated(ci, orbital_fragments, core_orbitals=ci.core_indices)

print("CI energy:           ", result["ci_energy"])
print("Decomposition energy:", result["decomposition_energy"])
print("Residual:            ", result["residual"])

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   N   0.00000000   0.00000000   0.00000000
   N   0.00000000   0.00000000   2.07435237
Parsed 2 atoms with basis set of 60 functions.
  Max eigenvalue: 4.073e+00
  Min eigenvalue: 1.153e-03
  Condition number: 3.534e+03
  Inverse condition number: 2.830e-04
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 60
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 1.153e-03
Number of electrons: 14
Number of alpha electrons: 7
Number of beta electrons: 7
Ms: 0
Total charge: 0
Number of basis functions: 60
Number of orthogonalized basis functions: 60
Number of auxiliary basis functions: 154
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.01 GB (doubled due to storing B_nPm)
Number of system basis functions: 60
Number of auxiliary basis function

In [24]:
frag_orbs = ci.active_indices

ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=ci.core_indices,
)

ecore = ints.E #core energy
print(ecore)
e1=0
e2=0
e3=0
e4=0

for i in frag_orbs:
    e1 += onefrag_correlation_energy_enumerated(ci, [i], core_orbitals=ci.core_indices)
    for j in frag_orbs:
        if j<i:
            e2 += twofrag_correlation_energy_enumerated(ci, [i], [j], core_orbitals=ci.core_indices)
            for k in frag_orbs:
                if k<j:
                    e3 += threefrag_correlation_energy_enumerated(ci, [i], [j], [k], core_orbitals=ci.core_indices)
                    for l in frag_orbs:
                        if l<k:
                            e4 += fourfrag_correlation_energy_enumerated(ci, [i], [j], [k], [l], core_orbitals=ci.core_indices)

print(f"Total one-orbital energy term: {e1} EH")
print(f"Total two-orbital energy term: {e2} EH")
print(f"Total three-orbital energy term: {e3} EH")
print(f"Total four-orbital energy term: {e4} EH")
print(f"Total energy from Decomposition Analysis: {ecore+e1+e2+e3+e4}")
print(f"Total CI energy: {ci.E[0]}")

-77.12007324912172
Total one-orbital energy term: -52.110849886694645 EH
Total two-orbital energy term: 20.239131958062217 EH
Total three-orbital energy term: -0.024595187215601264 EH
Total four-orbital energy term: -0.1031304977449989 EH
Total energy from Decomposition Analysis: -109.11951686271475
Total CI energy: -109.1195168627148


In [19]:
xyz = """
O 0.000 0.000 0.000
O 0.000 0.000 1.2075
"""
system = forte2.System(xyz=xyz, basis_set="cc-pVTZ", auxiliary_basis_set="def2-universal-JKFIT")

rhf = forte2.ROHF(charge=0,ms=1.0)(system)
rhf.run()

avas=forte2.AVAS(subspace=["O(2p)"],
                 selection_method="separate",
                 num_active_docc=3,
                 num_active_uocc=3)(rhf)

ci=forte2.CI(forte2.State(system=system, multiplicity=3,ms=1.0))(avas)
ci.run()

orbital_fragments = [[i] for i in ci.mo_space.active_indices]

result = fragment_decomposition_energy_enumerated(ci, orbital_fragments, core_orbitals=ci.core_indices)

print("CI energy:           ", result["ci_energy"])
print("Decomposition energy:", result["decomposition_energy"])
print("Residual:            ", result["residual"])

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   O   0.00000000   0.00000000   0.00000000
   O   0.00000000   0.00000000   2.28184430
Parsed 2 atoms with basis set of 60 functions.
  Max eigenvalue: 3.735e+00
  Min eigenvalue: 4.428e-03
  Condition number: 8.434e+02
  Inverse condition number: 1.186e-03
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 60
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 4.428e-03
Number of electrons: 16
Number of alpha electrons: 9
Number of beta electrons: 7
Ms: 1.0
Total charge: 0
Number of basis functions: 60
Number of orthogonalized basis functions: 60
Number of auxiliary basis functions: 154
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> ROHF SCF ROUTINE <==
Memory requirements: 0.01 GB (doubled due to storing B_nPm)
Number of system basis functions: 60
Number of auxiliary basis funct

In [20]:
frag_orbs = ci.active_indices

ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=ci.core_indices,
)

ecore = ints.E #core energy
print(ecore)
e1=0
e2=0
e3=0
e4=0

for i in frag_orbs:
    e1 += onefrag_correlation_energy_enumerated(ci, [i], core_orbitals=ci.core_indices)
    for j in frag_orbs:
        if j<i:
            e2 += twofrag_correlation_energy_enumerated(ci, [i], [j], core_orbitals=ci.core_indices)
            for k in frag_orbs:
                if k<j:
                    e3 += threefrag_correlation_energy_enumerated(ci, [i], [j], [k], core_orbitals=ci.core_indices)
                    for l in frag_orbs:
                        if l<k:
                            e4 += fourfrag_correlation_energy_enumerated(ci, [i], [j], [k], [l], core_orbitals=ci.core_indices)

print(f"Total one-orbital energy term: {e1} EH")
print(f"Total two-orbital energy term: {e2} EH")
print(f"Total three-orbital energy term: {e3} EH")
print(f"Total four-orbital energy term: {e4} EH")
print(f"Total energy from Decomposition Analysis: {ecore+e1+e2+e3+e4}")
print(f"Total CI energy: {ci.E[0]}")

-127.3784005849577
Total one-orbital energy term: -36.46622609546845 EH
Total two-orbital energy term: 14.217161720403649 EH
Total three-orbital energy term: 0.0008423573777146792 EH
Total four-orbital energy term: -0.12686134199190727 EH
Total energy from Decomposition Analysis: -149.7534839446367
Total CI energy: -149.75348394463668


In [21]:
xyz = """
C 0.000 0.000 0.000
C 0.000 0.000 1.2425
"""
system = forte2.System(xyz=xyz, basis_set="cc-pVTZ", auxiliary_basis_set="def2-universal-JKFIT")

rhf = forte2.RHF(charge=0)(system)
rhf.run()

avas=forte2.AVAS(subspace=["C(2s)","C(2p)"],
                 selection_method="separate",
                 num_active_docc=4,
                 num_active_uocc=4)(rhf)

ci=forte2.CI(forte2.State(system=system, multiplicity=1,ms=0.0))(avas)
ci.run()

orbital_fragments = [[i] for i in ci.mo_space.active_indices]

result = fragment_decomposition_energy_enumerated(ci, orbital_fragments, core_orbitals=ci.core_indices)

print("CI energy:           ", result["ci_energy"])
print("Decomposition energy:", result["decomposition_energy"])
print("Residual:            ", result["residual"])

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   C   0.00000000   0.00000000   0.00000000
   C   0.00000000   0.00000000   2.34798471
Parsed 2 atoms with basis set of 60 functions.
  Max eigenvalue: 4.155e+00
  Min eigenvalue: 9.140e-04
  Condition number: 4.546e+03
  Inverse condition number: 2.200e-04
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 60
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 9.140e-04
Number of electrons: 12
Number of alpha electrons: 6
Number of beta electrons: 6
Ms: 0
Total charge: 0
Number of basis functions: 60
Number of orthogonalized basis functions: 60
Number of auxiliary basis functions: 150
Energy convergence criterion: 1.000000e-09
Density convergence criterion: 1.000000e-06
DIIS acceleration: True

==> RHF SCF ROUTINE <==
Memory requirements: 0.01 GB (doubled due to storing B_nPm)
Number of system basis functions: 60
Number of auxiliary basis function

In [22]:
frag_orbs = ci.active_indices

ints = forte2.jkbuilder.RestrictedMOIntegrals(
    system=ci.system,
    C=ci.C[0],
    orbitals=frag_orbs,
    core_orbitals=ci.core_indices,
)

ecore = ints.E #core energy
print(ecore)
e1=0
e2=0
e3=0
e4=0

for i in frag_orbs:
    e1 += onefrag_correlation_energy_enumerated(ci, [i], core_orbitals=ci.core_indices)
    for j in frag_orbs:
        if j<i:
            e2 += twofrag_correlation_energy_enumerated(ci, [i], [j], core_orbitals=ci.core_indices)
            for k in frag_orbs:
                if k<j:
                    e3 += threefrag_correlation_energy_enumerated(ci, [i], [j], [k], core_orbitals=ci.core_indices)
                    for l in frag_orbs:
                        if l<k:
                            e4 += fourfrag_correlation_energy_enumerated(ci, [i], [j], [k], [l], core_orbitals=ci.core_indices)

print(f"Total one-orbital energy term: {e1} EH")
print(f"Total two-orbital energy term: {e2} EH")
print(f"Total three-orbital energy term: {e3} EH")
print(f"Total four-orbital energy term: {e4} EH")
print(f"Total energy from Decomposition Analysis: {ecore+e1+e2+e3+e4}")
print(f"Total CI energy: {ci.E[0]}")

-57.76712657311497
Total one-orbital energy term: -28.3525011841468 EH
Total two-orbital energy term: 10.701608901947795 EH
Total three-orbital energy term: -0.013992291732769667 EH
Total four-orbital energy term: -0.1073463379084747 EH
Total energy from Decomposition Analysis: -75.53935748495523
Total CI energy: -75.53935748495529
